<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/4_model_training/4_6_model_tcn_sub.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4_6_model_tcn

Temporal Convolutional Network, o Red Convolucional Temporal

## Introducción y Resumen

El objetivo de esta notebook ....



## 0. Configuración del Entorno


### 0.1. Instalación de librerías


In [21]:
# añadimos utilidades de torch
!pip -q install torchinfo einops --upgrade


### 0.2. Importación de librerías


In [22]:
import sys, platform, os, random
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torchinfo import summary

# Utilidades
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# ==============================
# Librerías estándar de Python
# ==============================
import os
import sys
import re
import glob
import warnings
import requests
from datetime import datetime, timedelta
from functools import reduce

# ==============================
# Manejo y procesamiento de datos
# ==============================
import pandas as pd
import numpy as np
from tabulate import tabulate

# ==============================
# Visualización
# ==============================
import matplotlib.pyplot as plt

# ==============================
# Estadística
# ==============================
from scipy.stats import spearmanr

# ==============================
# Machine Learning y utilidades
# ==============================
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.neural_network import MLPRegressor
import sklearn, scipy, numpy #, optuna

import joblib

# ==============================
# Configuración general
# ==============================
warnings.filterwarnings("ignore")

import time

import sys, platform, lightgbm as lgb
import numpy as np, pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score



In [23]:
print("python:", sys.version)
print("Platform:", platform.platform())
print("numpy:", numpy.__version__)
print("scipy:", scipy.__version__)
print("sklearn:", sklearn.__version__)
#print("optuna:", optuna.__version__)
#print("xgboost:", xgb.__version__)
#print("lightgbm:", lgb.__version__)

# CatBoost usa la clase para exponer versión
#print("catboost:", catboost.__version__)

python: 3.12.11 (main, Jun  4 2025, 08:56:18) [GCC 11.4.0]
Platform: Linux-6.6.97+-x86_64-with-glibc2.35
numpy: 2.0.2
scipy: 1.16.2
sklearn: 1.6.1


In [24]:
# Chequeo de GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Usando CPU")


CUDA disponible: True
GPU: Tesla T4


In [25]:
# Seeds para reproducibilidad
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

### 0.3. Acceso a Drive

In [26]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Mounted at /content/drive


## 1. Carga de datos

### 1.1. Carga de datasets `mnq_train`, `mnq_valid` y `mnq_test`






In [27]:
def load_data(data: str):

    data_path = f'{drive_path}/3_dataset_preparation/mnq_{data}.parquet'
    # Leer el archivo Parquet y cargarlo en un DataFrame
    df = pd.read_parquet(data_path)

    # Asegurar que el índice esté en formato datetime
    df.index = pd.to_datetime(df.index)

    # Crear una nueva columna 'date' con la fecha extraída del índice
    df['date'] = df.index.date

    # Reordenar columnas: 'date', 'time_str', y luego el resto
    cols = ['date'] + [col for col in df.columns if col not in ['date']]

    df = df[cols]

    return df

In [28]:
#mnq_train = load_data("train")
#mnq_valid = load_data("valid")
#mnq_test = load_data("test")

### 1.2. Información de datasets


In [29]:
def info_dataset(df, name: str):
  print(f"Información del dataset {name}:\n")

  # Contar valores únicos en la columna 'date'
  num_dias = df['date'].nunique()
  print(f"\tCantidad de días: {num_dias}")

  # Filtrar valores válidos
  validos_por_dia = df.dropna(subset=['close']).groupby('date').size()

  # Calcular el promedio
  promedio_por_fecha = validos_por_dia.mean()
  print(f"\tRegistros por día: {int(promedio_por_fecha)}")

  primer_hora = df.index[0].strftime('%H:%M')
  ultima_hora = df.index[-1].strftime('%H:%M')
  zona_horaria = df.index[0].tzinfo


  print(f"\tHora diaria de inicio {primer_hora}")
  print(f"\tHora diaria de final {ultima_hora}")
  print(f"\tZona horaria: {zona_horaria}\n")

  return num_dias, promedio_por_fecha

In [30]:
#info_dataset(mnq_train, 'mnq_train')
#info_dataset(mnq_valid, 'mnq_valid')
#info_dataset(mnq_test, 'mnq_test')

### 1.3. Carga de listado de features por ventana de tiempo

In [31]:
import json

# Ruta al archivo guardado
path = f'{drive_path}/2_feature_engineering/features_list.json'

with open(path, "r") as f:
    features_dict = json.load(f)

# Extraer las listas
features_to_30 = features_dict["features_to_30"]
features_to_60 = features_dict["features_to_60"]
features_to_90 = features_dict["features_to_90"]


In [32]:
print(f'Listado de features para 30min: {features_to_30}')
print(f'Listado de features para 60min: {features_to_60}')
print(f'Listado de features para 90min: {features_to_90}')

Listado de features para 30min: ['ire_90', 'rev_mom_z_90', 'roc_60', 'rev_score_90', 'price_ema30']
Listado de features para 60min: ['ire_60', 'rev_mom_z_90', 'roc_60', 'bb_60', 'rev_mom_vol_z_60', 'momentum_5', 'roc_20']
Listado de features para 90min: ['ire_60', 'rev_mom_z_90', 'roc_60', 'bb_60', 'momentum_5', 'roc_20', 'rev_mom_vol_z_60']


## 2. Carga de ventanas `X_train_*_scaled`, `X_valid_*_scaled`, `X_test_*_scaled`

### 2.0. Funciones

#### Función para cargar ventanas

In [33]:
def load_windows_and_scaler(target: str, scaled=True):
    """
    Carga datasets (X, y) para train, valid y test junto con el scaler global.

    Parámetros
    ----------
    drive_path : str
        Ruta base donde se encuentran los archivos.
    scaled : bool, default=True
        Si True busca en la carpeta 'ventanas_x_y_scaled',
        si False en 'ventanas_x_y'.

    Retorna
    -------
    X_train, y_train, X_valid, y_valid, X_test, y_test, scaler
    """

    #Ruta de ventandas escaladas
    path_train  = f'{drive_path}/3_dataset_preparation/xy_windows_scaled/xy_train_{target}_scaled.npz'
    path_valid  = f'{drive_path}/3_dataset_preparation/xy_windows_scaled/xy_valid_{target}_scaled.npz'
    path_test   = f'{drive_path}/3_dataset_preparation/xy_windows_scaled/xy_test_{target}_scaled.npz'

    #Ruta de escalador
    path_scaler = f"{drive_path}/3_dataset_preparation/global_scaler_{target}.pkl"

    # Cargar npz
    data_train = np.load(path_train)
    data_valid = np.load(path_valid)
    data_test  = np.load(path_test)

    # Extraer X, y
    X_train, y_train = data_train["X"], data_train["y"]
    X_valid, y_valid = data_valid["X"], data_valid["y"]
    X_test,  y_test  = data_test["X"],  data_test["y"]

    # Cargar scaler
    scaler = joblib.load(path_scaler)

    return X_train, y_train, X_valid, y_valid, X_test, y_test, scaler


#### Función para revisar información de ventanas

In [34]:
def xy_info(target: str, X_train, y_train, X_valid, y_valid, X_test, y_test):
    print(f'Información para horizonte de {target} minutos:')

    for name, X, y in [
        ("entrenamiento", X_train, y_train),
        ("validación", X_valid, y_valid),
        ("testeo", X_test, y_test),
    ]:
        print(f'\nSet de {name}:')
        print(f'\t{X.shape[0]} ventanas (n_samples).')

        if X.ndim == 2:

            print(f'\t{X.shape[1]} features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_{target})')
        elif X.ndim == 3:

            print(f'\t{X.shape[1]} pasos en lookback × {X.shape[2]} features por paso. Dimensión 3D: (n_samples, window_size, len(features_{target})')

        print(f'\t{y.shape[0]} targets.')
        print(f'\tDistribución y: mean={y.mean():.6f}, std={y.std():.6f}, min={y.min():.6f}, max={y.max():.6f}')

    return  X_train.shape[0], X_valid.shape[0], X_test.shape[0]

### 2.1 Carga de ventanas 30 minutos

In [35]:
X_train_30_scaled, y_train_30, X_valid_30_scaled, y_valid_30, X_test_30_scaled, y_test_30, scaler_30 = load_windows_and_scaler(target = '30')

In [36]:
n_samples_train_30, n_samples_valid_30, n_samples_test_30 = xy_info( '30', X_train_30_scaled, y_train_30, X_valid_30_scaled, y_valid_30, X_test_30_scaled, y_test_30)

Información para horizonte de 30 minutos:

Set de entrenamiento:
	193487 ventanas (n_samples).
	900 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_30)
	193487 targets.
	Distribución y: mean=0.000056, std=0.002760, min=-0.028931, max=0.032372

Set de validación:
	41567 ventanas (n_samples).
	900 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_30)
	41567 targets.
	Distribución y: mean=0.000103, std=0.003273, min=-0.023167, max=0.068056

Set de testeo:
	41567 ventanas (n_samples).
	900 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_30)
	41567 targets.
	Distribución y: mean=-0.000023, std=0.002407, min=-0.015982, max=0.016097


### 2.2 Carga de ventanas 60 minutos

In [37]:
X_train_60_scaled, y_train_60, X_valid_60_scaled, y_valid_60, X_test_60_scaled, y_test_60, scaler_60 = load_windows_and_scaler(target = '60')

In [38]:
n_samples_train_60, n_samples_valid_60, n_samples_test_60 = xy_info( '60', X_train_60_scaled, y_train_60, X_valid_60_scaled, y_valid_60, X_test_60_scaled, y_test_60)

Información para horizonte de 60 minutos:

Set de entrenamiento:
	193487 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_60)
	193487 targets.
	Distribución y: mean=0.000114, std=0.003907, min=-0.040179, max=0.036224

Set de validación:
	41567 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_60)
	41567 targets.
	Distribución y: mean=0.000163, std=0.004643, min=-0.038084, max=0.079896

Set de testeo:
	41567 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_60)
	41567 targets.
	Distribución y: mean=-0.000068, std=0.003521, min=-0.018494, max=0.020365


### 2.3 Carga de ventanas 90 minutos

In [39]:
X_train_90_scaled, y_train_90, X_valid_90_scaled, y_valid_90, X_test_90_scaled, y_test_90, scaler_90 = load_windows_and_scaler(target = '90')

In [40]:
n_samples_train_90, n_samples_valid_90, n_samples_test_90 = xy_info('90', X_train_90_scaled, y_train_90, X_valid_90_scaled, y_valid_90, X_test_90_scaled, y_test_90)

Información para horizonte de 90 minutos:

Set de entrenamiento:
	193487 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_90)
	193487 targets.
	Distribución y: mean=0.000159, std=0.004746, min=-0.037742, max=0.039284

Set de validación:
	41567 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_90)
	41567 targets.
	Distribución y: mean=0.000224, std=0.005887, min=-0.040748, max=0.083184

Set de testeo:
	41567 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_90)
	41567 targets.
	Distribución y: mean=-0.000132, std=0.004558, min=-0.023392, max=0.026213


## 3. Dataset de Métricas

Dado que cada entrenamiento demanda un tiempo considerable, antes de proceder verificaremos si ya existe un resultado previo de este modelo consultando el dataset de métricas.

### 3.1. Función para cargar métricas o generar dataset

In [41]:
def load_metrics(data: str):
    data_path = f'{drive_path}/4_model_training/{data}.parquet'
    # Leer el archivo Parquet y cargarlo en un DataFrame
    df = pd.read_parquet(data_path)
    return df

In [42]:
def metrics_verify(data: str) -> bool:
    data_path = f"{drive_path}/4_model_training/{data}.parquet"
    return os.path.exists(data_path)


In [43]:
def load_or_create_metrics (data:str):
  if metrics_verify(data):
      print(f"Las métricas existen y son almacenadas en {data[4:len(data)]}")
      model_metrics = load_metrics(data)
      #print(random_forest_metrics)
      metrics = True
  else:
      print(f"Las métricas no existen. Se crea el dataset {data[4:len(data)]} para almacenar las métricas")
      #Creamos la tabla para almacenar las métricas
      model_metrics = pd.DataFrame(columns=["RMSE", "MAE", "R2", "SMAPE", "DirAcc"])
      metrics = False

  return model_metrics, metrics

In [44]:
tcn_metrics, metrics = load_or_create_metrics("4_6_tcn_metrics")

Las métricas existen y son almacenadas en tcn_metrics


In [45]:
tcn_metrics

,RMSE,MAE,R2,SMAPE,DirAcc
TCN_30_subsampleado_30%,0.003732,0.002513,-0.351684,133.225328,0.588693
TCN_30_subsampleado_50%,0.004111,0.002526,-0.539696,133.397573,0.588414
TCN_30_100%,0.012594,0.008548,-13.811010,159.126102,0.503789
TCN_60_subsampleado_30%,0.004257,0.002588,0.109291,116.121781,0.688292
TCN_60_subsampleado_50%,0.004399,0.002561,0.089928,114.839184,0.698504
TCN_60_100%,0.012721,0.007892,-6.505905,142.887624,0.608247
TCN_90_subsampleado_30%,0.005707,0.003180,0.063686,111.166983,0.725662
TCN_90_subsampleado_50%,0.005412,0.002896,0.183344,104.811059,0.751480
TCN_90_100%,0.009756,0.007167,-1.746164,143.959714,0.588905


In [49]:
tcn_metrics_0 = tcn_metrics.copy() #Sin regularización de entrenamiento por dataset

### 3.2. Función para guardar métricas

In [54]:
def save_metrics (metrics,  metrics_name: str):   #("4_2_xgboost_metrics")
  metrics_path = f"{drive_path}/4_model_training/{metrics_name}.parquet"
  metrics.to_parquet(metrics_path, index = True)
  print(f"Métricas guardadas en {metrics_path}")

### 3.3. Función para calcular las métricas

In [55]:
def evaluate_model(model, X, y_true, y_pred=None, eps=1e-8):
    """
    Evalúa RMSE, MAE, R2, SMAPE y DirAcc.
    - Si y_pred es None, predice con el modelo usando X.
    - Evita mean_squared_error(squared=...) para máxima compatibilidad.
    """
    if y_pred is None:
        y_pred = model.predict(X)

    # Asegurar 1D
    y_true = np.ravel(y_true)
    y_pred = np.ravel(y_pred)

    # RMSE sin sklearn
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    mae = float(mean_absolute_error(y_true, y_pred))
    r2  = float(r2_score(y_true, y_pred))

    # SMAPE
    smape_val = 100.0 * np.mean(
        (np.abs(y_true - y_pred) / ((np.abs(y_true) + np.abs(y_pred)) / 2.0 + eps))
    )

    # Directional Accuracy
    directional_acc = float(np.mean(np.sign(y_true) == np.sign(y_pred)))

    return {
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "SMAPE": float(smape_val),
        "DirAcc": directional_acc
    }

In [56]:
def print_metrics(metrics, target:str):
  print(f"Métricas de {target}:\n")
  for k, v in metrics.items():
      print(f"\t{k:>5}:\t {float(v):.6f}")

## 4. Definición de modelo


### 4.1. Función de entrenamiento para modelo

In [57]:
from typing import Optional, Dict, Any, Tuple
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

In [58]:
'''
def train_model_tcn(
    best_params: Optional[Dict[str, Any]],
    X_train: np.ndarray, y_train: np.ndarray,
    X_valid: np.ndarray, y_valid: np.ndarray,
    *,
    use_internal_early_stopping: bool = True,
    # Arquitectura
    num_channels: Tuple[int, ...] = (64, 64, 64),
    kernel_size: int = 3,
    dropout: float = 0.1,
    use_weight_norm: bool = True,  # <-- NUEVO: para controlar weight_norm
    # Optimización
    lr: float = 1e-3,
    weight_decay: float = 0.0,
    batch_size: int = 512,
    max_epochs: int = 50,
    patience: int = 8,
    # Varios
    num_workers: int = 0,
    device: Optional[str] = None,
    verbose: bool = False,
):
    """
    Entrena un TCN (Temporal Convolutional Network) para regresión y devuelve (modelo, preds_valid).
    Requisitos:
      - X_* debe ser 3D: (n_samples, seq_len, n_features)
      - y_* puede ser 1D o 2D con última dimensión = 1
      - Los datos ya deben venir escalados si corresponde.
    """

    # -------------------------
    # Utilidades: módulos TCN
    # -------------------------
    class CausalConv1d(nn.Module):
        def __init__(self, in_ch, out_ch, k, dilation=1, use_weight_norm=True):
            super().__init__()
            self.pad = (k - 1) * dilation
            self.padding = nn.ConstantPad1d((self.pad, 0), 0.0)
            conv = nn.Conv1d(in_ch, out_ch, kernel_size=k, dilation=dilation)
            # APLICAMOS weight_norm AL CONV REAL (NO AL WRAPPER)
            self.conv = nn.utils.weight_norm(conv) if use_weight_norm else conv

        def forward(self, x):
            # x: (B, C, T)
            return self.conv(self.padding(x))  # salida causal

    class TemporalBlock(nn.Module):
        def __init__(self, in_ch, out_ch, k, dilation, dropout, use_weight_norm=True):
            super().__init__()
            self.conv1 = CausalConv1d(in_ch, out_ch, k, dilation, use_weight_norm)
            self.relu1 = nn.ReLU()
            self.drop1 = nn.Dropout(dropout)

            self.conv2 = CausalConv1d(out_ch, out_ch, k, dilation, use_weight_norm)
            self.relu2 = nn.ReLU()
            self.drop2 = nn.Dropout(dropout)

            self.downsample = nn.Conv1d(in_ch, out_ch, kernel_size=1) if in_ch != out_ch else None
            self.final_relu = nn.ReLU()

        def forward(self, x):
            out = self.drop1(self.relu1(self.conv1(x)))
            out = self.drop2(self.relu2(self.conv2(out)))
            res = x if self.downsample is None else self.downsample(x)
            return self.final_relu(out + res)

    class TemporalConvNet(nn.Module):
        def __init__(self, in_channels, num_channels, k=3, dropout=0.1, use_weight_norm=True):
            super().__init__()
            layers = []
            prev_ch = in_channels
            for i, ch in enumerate(num_channels):
                dilation = 2 ** i
                layers.append(TemporalBlock(prev_ch, ch, k, dilation, dropout, use_weight_norm))
                prev_ch = ch
            self.network = nn.Sequential(*layers)

        def forward(self, x):
            # x: (B, C_in, T) -> (B, C_last, T)
            return self.network(x)

    class TCNRegressor(nn.Module):
        def __init__(self, in_features, num_channels, k=3, dropout=0.1, use_weight_norm=True):
            super().__init__()
            self.tcn = TemporalConvNet(in_channels=in_features,
                                       num_channels=num_channels,
                                       k=k,
                                       dropout=dropout,
                                       use_weight_norm=use_weight_norm)
            self.head = nn.Linear(num_channels[-1], 1)  # tomamos el último paso temporal

        def forward(self, x):
            # x: (B, T, F) -> (B, F, T)
            x = x.transpose(1, 2)
            y = self.tcn(x)              # (B, C_last, T)
            last = y[:, :, -1]           # (B, C_last) - último tiempo
            out = self.head(last)        # (B, 1)
            return out.squeeze(-1)       # (B,)

    # -------------------------
    # Defaults + overrides
    # -------------------------
    params = dict(best_params or {})
    num_channels = tuple(params.get("num_channels", num_channels))
    kernel_size = int(params.get("kernel_size", kernel_size))
    dropout = float(params.get("dropout", dropout))
    lr = float(params.get("lr", lr))
    weight_decay = float(params.get("weight_decay", weight_decay))
    batch_size = int(params.get("batch_size", batch_size))
    max_epochs = int(params.get("max_epochs", max_epochs))
    patience = int(params.get("patience", patience))
    use_weight_norm = bool(params.get("use_weight_norm", use_weight_norm))

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    device = torch.device(device)

    # -------------------------
    # Validaciones de forma
    # -------------------------
    if X_train.ndim != 3 or X_valid.ndim != 3:
        raise ValueError("X_train y X_valid deben ser 3D: (n_samples, seq_len, n_features).")

    # y reshape
    y_train = y_train.reshape(-1)
    y_valid = y_valid.reshape(-1)

    n_features = X_train.shape[-1]

    # -------------------------
    # Tensores y loaders
    # -------------------------
    Xtr = torch.tensor(X_train, dtype=torch.float32)
    Ytr = torch.tensor(y_train, dtype=torch.float32)
    Xva = torch.tensor(X_valid, dtype=torch.float32)
    Yva = torch.tensor(y_valid, dtype=torch.float32)

    train_ds = TensorDataset(Xtr, Ytr)
    valid_ds = TensorDataset(Xva, Yva)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    valid_loader = DataLoader(valid_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)

    # -------------------------
    # Modelo, criterio, optim
    # -------------------------
    model = TCNRegressor(in_features=n_features,
                         num_channels=num_channels,
                         k=kernel_size,
                         dropout=dropout,
                         use_weight_norm=use_weight_norm).to(device)

    criterion = torch.nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    # -------------------------
    # Entrenamiento + Early Stop
    # -------------------------
    best_state = None
    best_val = float("inf")
    epochs_no_improve = 0

    for epoch in range(max_epochs):
        model.train()
        train_loss = 0.0
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * xb.size(0)

        train_loss /= len(train_ds)

        # Validación
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in valid_loader:
                xb = xb.to(device)
                yb = yb.to(device)
                preds = model(xb)
                loss = criterion(preds, yb)
                val_loss += loss.item() * xb.size(0)
        val_loss /= len(valid_ds)

        if verbose:
            print(f"Epoch {epoch+1:03d}/{max_epochs} | train_loss={train_loss:.6f} | val_loss={val_loss:.6f}")

        # Early stopping
        if use_internal_early_stopping:
            if val_loss < best_val - 1e-10:
                best_val = val_loss
                epochs_no_improve = 0
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= patience:
                    if verbose:
                        print(f"Early stopping en epoch {epoch+1} (mejor val_loss={best_val:.6f}).")
                    break
        else:
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    # Restaurar mejor estado
    if best_state is not None:
        model.load_state_dict(best_state)

    # -------------------------
    # Predicciones valid
    # -------------------------
    model.eval()
    preds_list = []
    with torch.no_grad():
        for xb, _ in valid_loader:
            xb = xb.to(device)
            preds = model(xb).detach().cpu().numpy()
            preds_list.append(preds)
    preds_valid = np.concatenate(preds_list, axis=0)

    return model, preds_valid
    '''

### Aficional


In [105]:
from typing import Optional, Dict, Any, Tuple
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

def train_model_tcn(
    best_params: Optional[Dict[str, Any]],
    X_train: np.ndarray, y_train: np.ndarray,
    X_valid: np.ndarray, y_valid: np.ndarray,
    *,
    use_internal_early_stopping: bool = True,
    # Arquitectura
    num_channels: Tuple[int, ...] = (64, 64, 64),
    kernel_size: int = 3,
    dropout: float = 0.1,
    use_weight_norm: bool = True,
    # Optimización
    lr: float = 1e-3,
    weight_decay: float = 0.0,
    batch_size: int = 512,
    max_epochs: int = 50,
    patience: int = 8,
    # Estabilizadores
    grad_clip: Optional[float] = 1.0,          # None para desactivar clipping
    use_scheduler: bool = True,                # ReduceLROnPlateau
    scheduler_factor: float = 0.5,             # factor de reducción del LR
    scheduler_patience: int = 3,               # epochs sin mejora para reducir LR
    scheduler_min_lr: float = 1e-5,            # LR mínimo
    # Varios
    num_workers: int = 0,
    device: Optional[str] = None,
    verbose: bool = False,
):
    """
    Entrena un TCN para regresión y devuelve (modelo, preds_valid).
    - X_* debe ser 3D: (n_samples, seq_len, n_features)
    - y_* puede ser 1D o 2D con última dimensión = 1
    - Incluye gradient clipping y ReduceLROnPlateau opcionales.
    """

    # -------------------------
    # Módulos TCN
    # -------------------------
    class CausalConv1d(nn.Module):
        def __init__(self, in_ch, out_ch, k, dilation=1, use_weight_norm=True):
            super().__init__()
            self.pad = (k - 1) * dilation
            self.padding = nn.ConstantPad1d((self.pad, 0), 0.0)
            conv = nn.Conv1d(in_ch, out_ch, kernel_size=k, dilation=dilation)
            self.conv = nn.utils.weight_norm(conv) if use_weight_norm else conv

        def forward(self, x):
            return self.conv(self.padding(x))  # (B, C_out, T) causal

    class TemporalBlock(nn.Module):
        def __init__(self, in_ch, out_ch, k, dilation, dropout, use_weight_norm=True):
            super().__init__()
            self.conv1 = CausalConv1d(in_ch, out_ch, k, dilation, use_weight_norm)
            self.relu1 = nn.ReLU()
            self.drop1 = nn.Dropout(dropout)

            self.conv2 = CausalConv1d(out_ch, out_ch, k, dilation, use_weight_norm)
            self.relu2 = nn.ReLU()
            self.drop2 = nn.Dropout(dropout)

            self.downsample = nn.Conv1d(in_ch, out_ch, kernel_size=1) if in_ch != out_ch else None
            self.final_relu = nn.ReLU()

        def forward(self, x):
            out = self.drop1(self.relu1(self.conv1(x)))
            out = self.drop2(self.relu2(self.conv2(out)))
            res = x if self.downsample is None else self.downsample(x)
            return self.final_relu(out + res)

    class TemporalConvNet(nn.Module):
        def __init__(self, in_channels, num_channels, k=3, dropout=0.1, use_weight_norm=True):
            super().__init__()
            layers = []
            prev_ch = in_channels
            for i, ch in enumerate(num_channels):
                dilation = 2 ** i
                layers.append(TemporalBlock(prev_ch, ch, k, dilation, dropout, use_weight_norm))
                prev_ch = ch
            self.network = nn.Sequential(*layers)

        def forward(self, x):
            return self.network(x)  # (B, C_last, T)

    class TCNRegressor(nn.Module):
        def __init__(self, in_features, num_channels, k=3, dropout=0.1, use_weight_norm=True):
            super().__init__()
            self.tcn = TemporalConvNet(in_channels=in_features,
                                       num_channels=num_channels,
                                       k=k,
                                       dropout=dropout,
                                       use_weight_norm=use_weight_norm)
            self.head = nn.Linear(num_channels[-1], 1)

        def forward(self, x):
            # x: (B, T, F) -> (B, F, T)
            x = x.transpose(1, 2)
            y = self.tcn(x)          # (B, C_last, T)
            last = y[:, :, -1]       # último paso temporal
            out = self.head(last)    # (B, 1)
            return out.squeeze(-1)   # (B,)

    # -------------------------
    # Defaults + overrides
    # -------------------------
    params = dict(best_params or {})
    num_channels   = tuple(params.get("num_channels", num_channels))
    kernel_size    = int(params.get("kernel_size", kernel_size))
    dropout        = float(params.get("dropout", dropout))
    lr             = float(params.get("lr", lr))
    weight_decay   = float(params.get("weight_decay", weight_decay))
    batch_size     = int(params.get("batch_size", batch_size))
    max_epochs     = int(params.get("max_epochs", max_epochs))
    patience       = int(params.get("patience", patience))
    use_weight_norm = bool(params.get("use_weight_norm", use_weight_norm))

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    device = torch.device(device)

    # -------------------------
    # Validaciones de forma
    # -------------------------
    if X_train.ndim != 3 or X_valid.ndim != 3:
        raise ValueError("X_train y X_valid deben ser 3D: (n_samples, seq_len, n_features).")

    y_train = y_train.reshape(-1)
    y_valid = y_valid.reshape(-1)

    n_features = X_train.shape[-1]

    # -------------------------
    # Tensores y loaders
    # -------------------------
    Xtr = torch.tensor(X_train, dtype=torch.float32)
    Ytr = torch.tensor(y_train, dtype=torch.float32)
    Xva = torch.tensor(X_valid, dtype=torch.float32)
    Yva = torch.tensor(y_valid, dtype=torch.float32)

    train_ds = TensorDataset(Xtr, Ytr)
    valid_ds = TensorDataset(Xva, Yva)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    valid_loader = DataLoader(valid_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)

    # -------------------------
    # Modelo, criterio, optim, scheduler
    # -------------------------
    model = TCNRegressor(in_features=n_features,
                         num_channels=num_channels,
                         k=kernel_size,
                         dropout=dropout,
                         use_weight_norm=use_weight_norm).to(device)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    scheduler = None
    if use_scheduler:
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", factor=scheduler_factor,
            patience=scheduler_patience, min_lr=scheduler_min_lr)# verbose=verbose        )

    # -------------------------
    # Entrenamiento + Early Stop
    # -------------------------
    best_state = None
    best_val = float("inf")
    epochs_no_improve = 0

    for epoch in range(max_epochs):
        model.train()
        train_loss = 0.0

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad(set_to_none=True)
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()

            # ✅ Gradient clipping
            if grad_clip is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)

            optimizer.step()
            train_loss += loss.item() * xb.size(0)

        train_loss /= len(train_ds)

        # Validación
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in valid_loader:
                xb = xb.to(device)
                yb = yb.to(device)
                preds = model(xb)
                loss = criterion(preds, yb)
                val_loss += loss.item() * xb.size(0)
        val_loss /= len(valid_ds)

        if verbose:
            print(f"Epoch {epoch+1:03d}/{max_epochs} | train_loss={train_loss:.6f} | val_loss={val_loss:.6f}")

        # ✅ Scheduler (ReduceLROnPlateau)
        if scheduler is not None:
            scheduler.step(val_loss)

        # Early stopping
        if use_internal_early_stopping:
            if val_loss < best_val - 1e-10:
                best_val = val_loss
                epochs_no_improve = 0
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= patience:
                    if verbose:
                        print(f"Early stopping en epoch {epoch+1} (mejor val_loss={best_val:.6f}).")
                    break
        else:
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    # Restaurar mejor estado
    if best_state is not None:
        model.load_state_dict(best_state)

    # -------------------------
    # Predicciones valid
    # -------------------------
    model.eval()
    preds_list = []
    with torch.no_grad():
        for xb, _ in valid_loader:
            xb = xb.to(device)
            preds = model(xb).detach().cpu().numpy()
            preds_list.append(preds)
    preds_valid = np.concatenate(preds_list, axis=0)

    return model, preds_valid


### 4.2. Parámetros por defecto para modelo


In [96]:
import math
def tcn_params_for_setup(
    horizon_minutes: int,        # 30, 60 o 90 (horizonte de predicción)
    rate_sampled: float,         # 0.3, 0.5 o 1.0 (porcentaje del dataset)
    base_channels: int = 64,
    k: int = 3,
    window_size: int = 90,       # SIEMPRE 90 (ventana de entrada)
    rf_margin: float = 1.05,
    depth_cap: int = 8,
):
    """
    Devuelve un dict de hiperparámetros TCN ajustado según:
    - Horizonte de predicción (30, 60, 90)
    - Porcentaje de dataset usado (0.3, 0.5, 1.0)

    El campo receptivo (RF) se calcula sobre window_size=90 (entrada fija).
    """

    # ======================
    # 1) Calcular profundidad (L)
    # ======================
    target = int(math.ceil(window_size * rf_margin))
    rhs = max(1, (target - 1) / max(1, (k - 1)))     # (2^L - 1) >= (target - 1)/(k-1)
    L = int(math.ceil(math.log2(rhs + 1)))
    L = max(1, min(L, depth_cap))
    num_channels = tuple([base_channels] * L)

    # ======================
    # 2) Ajustes por rate_sampled
    # ======================
    if rate_sampled <= 0.3 + 1e-9:
        lr = 1e-3
        batch_size = 512
        dropout = 0.10
        weight_decay = 1e-5
        max_epochs = 60
        patience = 10
        use_weight_norm = True,
        verbose =  False

    elif rate_sampled <= 0.5 + 1e-9:
        lr = 7.5e-4
        batch_size = 384
        dropout = 0.15
        weight_decay = 1e-5
        max_epochs = 70
        patience = 12
        use_weight_norm = True,
        verbose =  False

    else:  # 100%
        lr = 3e-4
        batch_size = 256
        dropout = 0.25
        weight_decay = 1e-4
        max_epochs = 100
        patience = 15
        use_weight_norm = False,  # clave para estabilizar
        verbose =  False

    # ======================
    # 3) Microajuste por horizonte
    # ======================
    # horizontes más largos tienden a necesitar menos regularización
    if horizon_minutes >= 90:
        dropout = max(0.10, dropout - 0.05)

    # ======================
    # 4) Devolver configuración
    # ======================
    return {
        "num_channels": num_channels,
        "kernel_size": k,
        "dropout": dropout,
        "lr": lr,
        "weight_decay": weight_decay,
        "batch_size": batch_size,
        "max_epochs": max_epochs,
        "patience": patience,
        "use_weight_norm": True,
        "verbose": False
    }

In [60]:
tcn_params_30_30 = tcn_params_for_setup(horizon_minutes=30, rate_sampled=0.3)
tcn_params_30_50 = tcn_params_for_setup(horizon_minutes=30, rate_sampled=0.5)
tcn_params_30_100 = tcn_params_for_setup(horizon_minutes=30, rate_sampled=1.0)

tcn_params_60_30 = tcn_params_for_setup(horizon_minutes=60, rate_sampled=0.3)
tcn_params_60_50 = tcn_params_for_setup(horizon_minutes=60, rate_sampled=0.5)
tcn_params_60_100 = tcn_params_for_setup(horizon_minutes=60, rate_sampled=1.0)

tcn_params_90_30 = tcn_params_for_setup(horizon_minutes=90, rate_sampled=0.3)
tcn_params_90_50 = tcn_params_for_setup(horizon_minutes=90, rate_sampled=0.5)
tcn_params_90_100 = tcn_params_for_setup(horizon_minutes=90, rate_sampled=1.0)




### 4.3. Función conjunta

In [61]:
# Asumimos que ya definiste:
# - subsample(X, y, n)
# - train_model_tcn(best_params, X_train, y_train, X_valid, y_valid, ...)
# - evaluate_model(model, X, y_true, y_pred=None, eps=1e-8)  (opcional)

def _internal_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """Fallback si no pasás evaluate_fn. Calcula RMSE, MAE, R2, SMAPE y DirAcc."""
    from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae  = float(mean_absolute_error(y_true, y_pred))
    r2   = float(r2_score(y_true, y_pred))
    # SMAPE clásico (en %) con epsilon para evitar div/0
    eps = 1e-12
    denom = (np.abs(y_true) + np.abs(y_pred)).clip(min=eps)
    smape = float(np.mean(2.0 * np.abs(y_pred - y_true) / denom) * 100.0)
    diracc = float(np.mean(np.sign(y_pred) == np.sign(y_true)))
    return {"RMSE": rmse, "MAE": mae, "R2": r2, "SMAPE": smape, "DirAcc": diracc}


def run_tcn_experiment(
    metrics_flag: bool,
    model_key: str,
    X_train_scaled: np.ndarray,
    y_train: np.ndarray,
    X_valid_scaled: np.ndarray,
    y_valid: np.ndarray,
    resample_rate: float,                 # 0.3, 0.5, 1.0
    tcn_params: Dict[str, Any],
    tcn_metrics_df: Optional[pd.DataFrame] = None,
    n_samples_train: Optional[int] = None,
    n_samples_valid: Optional[int] = None,
    *,
    # extras específicos de TCN / PyTorch
    use_internal_early_stopping: bool = True,
    max_epochs: int = 50,
    patience: int = 8,
    verbose: bool = False,
    # soporte opcional para datos 2D → 3D
    enforce_3d_shape: Optional[Tuple[int, int]] = None,
    # hooks opcionales
    evaluate_fn=evaluate_model,          # si tenés tu evaluate_model(model, X, y, y_pred=...) pasalo acá
    print_metrics_fn=None      # si tenés tu print_metrics(dict, model_key) pasalo acá
) -> dict:
    """
    Ejecuta un experimento TCN con (opcional) subsampleo de train/valid.

    Parámetros
    ----------
    metrics_flag : bool
        Si True, no entrena y busca métricas previas en tcn_metrics_df[model_key].
    model_key : str
        Nombre/índice del modelo en la tabla de métricas (ej: 'TCN_60_subsampleado_50%').
    X_train_scaled, y_train : arrays
        Ventanas y target de entrenamiento (ya escaladas).
        Formato esperado: 3D (n_samples, seq_len, n_features).
        Si vienen 2D (n_samples, seq_len*n_features), podés usar enforce_3d_shape=(seq_len, n_features).
    X_valid_scaled, y_valid : arrays
        Ventanas y target de validación (ya escaladas), idem formato.
    resample_rate : float
        Proporción a muestrear (0 < r <= 1). 1.0 = sin subsampleo.
    tcn_params : dict
        Hiperparámetros nativos del TCN (num_channels, kernel_size, dropout, lr, etc.).
    tcn_metrics_df : pd.DataFrame | None
        DataFrame de métricas para leer/escribir (index por model_key). Opcional.
    n_samples_train, n_samples_valid : int | None
        Tamaños base para calcular la cantidad a muestrear. Si es None, se infiere de X_*.
    enforce_3d_shape : (window_size, n_features) | None
        Si tus X_* están 2D (aplanadas), se re-forman a 3D con este shape.

    Retorna
    -------
    dict
        {"RMSE","MAE","R2","SMAPE","DirAcc"}.
    """
    # 0) Carga de métricas previas (si corresponde)
    if metrics_flag and (tcn_metrics_df is not None) and (model_key in tcn_metrics_df.index):
        print("El modelo fue entrenado anteriormente y las métricas ya fueron calculadas\n")
        metrics_dict = tcn_metrics_df.loc[model_key].to_dict()
        if print_metrics_fn is not None:
            print_metrics_fn(metrics_dict, model_key)
        else:
            print(f"[{model_key}] -> {metrics_dict}")
        return metrics_dict

    # 1) Mensaje de entrenamiento
    tag_rate = f"{int(resample_rate*100)}%" if resample_rate < 1.0 else "100%"
    print(f"Entrenando modelo {model_key} (resample={tag_rate})...")

    # 2) Re-shape opcional 2D → 3D
    def _ensure_3d(X: np.ndarray, name: str) -> np.ndarray:
        if X.ndim == 3:
            return X
        if X.ndim == 2 and enforce_3d_shape is not None:
            T, F = enforce_3d_shape
            if X.shape[1] != T * F:
                raise ValueError(f"{name} tiene shape {X.shape} pero enforce_3d_shape={enforce_3d_shape} "
                                 f"no coincide con columnas={T*F}.")
            return X.reshape(X.shape[0], T, F)
        raise ValueError(f"{name} debe ser 3D (n, T, F) o pasar enforce_3d_shape=(T, F) si viene aplanado.")

    X_train_3d = _ensure_3d(X_train_scaled, "X_train_scaled")
    X_valid_3d = _ensure_3d(X_valid_scaled, "X_valid_scaled")

    # 3) Tamaños base
    if n_samples_train is None:
        n_samples_train = X_train_3d.shape[0]
    if n_samples_valid is None:
        n_samples_valid = X_valid_3d.shape[0]

    # 4) Subsampleo (si aplica)
    if resample_rate < 1.0:
        n_train_sub = max(1, int(n_samples_train * resample_rate))
        n_valid_sub = max(1, int(n_samples_valid * resample_rate))
        X_train_sub, y_train_sub = subsample(X_train_3d, y_train, n_train_sub)
        X_valid_sub, y_valid_sub = subsample(X_valid_3d, y_valid, n_valid_sub)
    else:
        X_train_sub, y_train_sub = X_train_3d, y_train
        X_valid_sub, y_valid_sub = X_valid_3d, y_valid

    # 5) Entrenamiento y predicción en valid
    model, y_pred_valid = train_model_tcn(
        best_params=tcn_params,
        X_train=X_train_sub, y_train=y_train_sub,
        X_valid=X_valid_sub, y_valid=y_valid_sub,
        use_internal_early_stopping=use_internal_early_stopping,
        max_epochs=max_epochs,
        patience=patience,
        verbose=verbose
    )

    # 6) Evaluación (usa tu evaluate_model si lo pasaste; si no, usa fallback interno)
    if evaluate_fn is not None:
        # Pasamos y_pred para no depender de que el modelo tenga .predict()
        metrics_dict = evaluate_fn(model, X_valid_sub, y_valid_sub, y_pred=y_pred_valid)
    else:
        metrics_dict = _internal_metrics(np.ravel(y_valid_sub), np.ravel(y_pred_valid))

    # 7) Mostrar
    if print_metrics_fn is not None:
        print_metrics_fn(metrics_dict, model_key)
    else:
        print(f"[{model_key}] -> {metrics_dict}")

    # 8) Persistir métricas en el DataFrame si se pasa
    if tcn_metrics_df is not None:
        tcn_metrics_df.loc[model_key] = metrics_dict

    return metrics_dict

### 4.4. Función para subsamplear

In [62]:
def subsample(X, y, n):
    n = min(n, X.shape[0])
    idx = np.random.choice(X.shape[0], size=n, replace=False)
    return X[idx], y[idx]

In [79]:
#xgb_device_test = xgb.XGBRegressor(tree_method="gpu_hist", predictor="gpu_predictor")
#try:
#    xgb_device_test.fit([[0,0],[1,1]], [0,1])
#    print("✅ XGBoost GPU works correctly")
#except Exception as e:
#    print("❌ GPU not available for XGBoost:", e)

## 5. Entrenamiento

### 5.1. Entrenamiento 30min

#### 5.1.1. Con 30% de dataset

In [64]:
windows_size = 90
n_samples_train_30
len(features_to_30)
features_to_30

['ire_90', 'rev_mom_z_90', 'roc_60', 'rev_score_90', 'price_ema30']

In [63]:
features_base = ['open','high','close','low','volume']

In [67]:
metrics = False

In [77]:
tcn_30_30 = run_tcn_experiment(
    metrics_flag=metrics,
    model_key="TCN_30_subsampleado_30%",
    X_train_scaled=X_train_30_scaled, y_train=y_train_30,
    X_valid_scaled=X_valid_30_scaled, y_valid=y_valid_30,
    resample_rate=0.3,                       # 30% del dataset
    tcn_params=tcn_params_30_30,           # hiperparámetros base
    tcn_metrics_df=tcn_metrics,              # DataFrame donde guardar métricas
    use_internal_early_stopping=True,        # activa early stopping
    #max_epochs=tcn_params_30_30["max_epochs"],                           # podés subirlo un poco
    #patience=tcn_params_30_30["patience"],                             # paciencia del early stopping
    verbose=False,                           # logs de entrenamiento

    # Si tus ventanas están aplanadas (2D), pasá su forma original:
    # enforce_3d_shape=(window_size, n_features)
    enforce_3d_shape=(90, len(features_to_30)+len(features_base)), # ejemplo si window_size=90

    # Hooks genéricos:
    evaluate_fn=evaluate_model,              # misma función de métricas
    print_metrics_fn=print_metrics           # tu función para mostrar métricas
)
#Time 262 segundos.

Entrenando modelo TCN_30_subsampleado_30% (resample=30%)...
Métricas de TCN_30_subsampleado_30%:

	 RMSE:	 0.003655
	  MAE:	 0.002355
	   R2:	 -0.306041
	SMAPE:	 130.023726
	DirAcc:	 0.611708


#### 5.1.2. Con 50% de dataset

In [82]:
tcn_params_30_50

{'num_channels': (64, 64, 64, 64, 64, 64),
 'kernel_size': 3,
 'dropout': 0.15,
 'lr': 0.00075,
 'weight_decay': 1e-05,
 'batch_size': 384,
 'max_epochs': 70,
 'patience': 12,
 'use_weight_norm': True,
 'verbose': False}

In [83]:
tcn_30_50 = run_tcn_experiment(
    metrics_flag=metrics,
    model_key="TCN_30_subsampleado_50%",
    X_train_scaled=X_train_30_scaled, y_train=y_train_30,
    X_valid_scaled=X_valid_30_scaled, y_valid=y_valid_30,
    resample_rate=0.5,                       # 30% del dataset
    tcn_params=tcn_params_30_50,           # hiperparámetros base
    tcn_metrics_df=tcn_metrics,              # DataFrame donde guardar métricas
    use_internal_early_stopping=True,        # activa early stopping
    #max_epochs=80,                           # podés subirlo un poco
    #patience=13,                             # paciencia del early stopping
    verbose=False,                           # logs de entrenamiento

    # Si tus ventanas están aplanadas (2D), pasá su forma original:
    # enforce_3d_shape=(window_size, n_features)
    enforce_3d_shape=(90, len(features_to_30)+len(features_base)), # ejemplo si window_size=90

    # Hooks genéricos:
    evaluate_fn=evaluate_model,              # misma función de métricas
    print_metrics_fn=print_metrics           # tu función para mostrar métricas
)
#Tiempo: 286s /

Entrenando modelo TCN_30_subsampleado_50% (resample=50%)...
Métricas de TCN_30_subsampleado_50%:

	 RMSE:	 0.003750
	  MAE:	 0.002498
	   R2:	 -0.343409
	SMAPE:	 131.578467
	DirAcc:	 0.601164


#### 5.1.3. Con ventanas completas

In [101]:
T = 90
F_30 = X_train_30_scaled.shape[1] // T
F_30

10

In [103]:
tcn_params_30_100

{'num_channels': (64, 64, 64, 64, 64, 64),
 'kernel_size': 3,
 'dropout': 0.2,
 'lr': 0.0005,
 'weight_decay': 0.0001,
 'batch_size': 256,
 'max_epochs': 80,
 'patience': 12,
 'use_weight_norm': False,
 'verbose': False}

In [102]:
tcn_params_30_100["use_weight_norm"] = False

In [ ]:
tcn_30_100 = run_tcn_experiment(
    metrics_flag=False,
    model_key="TCN_30_100%",
    X_train_scaled=X_train_30_scaled, y_train=y_train_30,
    X_valid_scaled=X_valid_30_scaled, y_valid=y_valid_30,
    resample_rate=1.0,
    tcn_params=tcn_params_30_100,
    tcn_metrics_df=tcn_metrics,
    use_internal_early_stopping=True,
    max_epochs=tcn_params_30_100["max_epochs"],
    patience=tcn_params_30_100["patience"],
    verbose=tcn_params_30_100["verbose"],
    enforce_3d_shape=(90, len(features_to_30)+len(features_base)),
    evaluate_fn=evaluate_model,
    print_metrics_fn=print_metrics
)

Entrenando modelo TCN_30_100% (resample=100%)...


In [86]:
tcn_metrics

,RMSE,MAE,R2,SMAPE,DirAcc
TCN_30_subsampleado_30%,0.003655,0.002355,-0.306041,130.023726,0.611708
TCN_30_subsampleado_50%,0.003750,0.002498,-0.343409,131.578467,0.601164
TCN_30_100%,0.012346,0.009199,-13.233888,158.551167,0.581471
TCN_60_subsampleado_30%,0.004257,0.002588,0.109291,116.121781,0.688292
TCN_60_subsampleado_50%,0.004399,0.002561,0.089928,114.839184,0.698504
TCN_60_100%,0.012721,0.007892,-6.505905,142.887624,0.608247
TCN_90_subsampleado_30%,0.005707,0.003180,0.063686,111.166983,0.725662
TCN_90_subsampleado_50%,0.005412,0.002896,0.183344,104.811059,0.751480
TCN_90_100%,0.009756,0.007167,-1.746164,143.959714,0.588905


### 4.2. Entrenamiento 60min

#### 4.2.1. Con 30% de dataset

In [88]:
tcn_60_30 = run_tcn_experiment(
    metrics_flag=metrics,
    model_key="TCN_60_subsampleado_30%",
    X_train_scaled=X_train_60_scaled, y_train=y_train_60,
    X_valid_scaled=X_valid_60_scaled, y_valid=y_valid_60,
    resample_rate=0.3,                       # 30% del dataset
    tcn_params=tcn_params_60_30,           # hiperparámetros base
    tcn_metrics_df=tcn_metrics,              # DataFrame donde guardar métricas
    use_internal_early_stopping=True,        # activa early stopping
    #max_epochs=tcn_params_60_30["max_epochs"],                           # podés subirlo un poco
    #patience=tcn_params_60_30["patience"],                             # paciencia del early stopping
    verbose=False,                           # logs de entrenamiento

    # Si tus ventanas están aplanadas (2D), pasá su forma original:
    # enforce_3d_shape=(window_size, n_features)
    enforce_3d_shape=(90, len(features_to_60)+len(features_base)), # ejemplo si window_size=90

    # Hooks genéricos:
    evaluate_fn=evaluate_model,              # misma función de métricas
    print_metrics_fn=print_metrics           # tu función para mostrar métricas
)
#Time: 159s

Entrenando modelo TCN_60_subsampleado_30% (resample=30%)...
Métricas de TCN_60_subsampleado_30%:

	 RMSE:	 0.005542
	  MAE:	 0.003168
	   R2:	 -0.445459
	SMAPE:	 126.158065
	DirAcc:	 0.643705


#### 4.2.2. Con 50% de dataset

In [89]:
tcn_60_50 = run_tcn_experiment(
    metrics_flag=metrics,
    model_key="TCN_60_subsampleado_50%",
    X_train_scaled=X_train_60_scaled, y_train=y_train_60,
    X_valid_scaled=X_valid_60_scaled, y_valid=y_valid_60,
    resample_rate=0.5,                       # 30% del dataset
    tcn_params=tcn_params_60_50,           # hiperparámetros base
    tcn_metrics_df=tcn_metrics,              # DataFrame donde guardar métricas
    use_internal_early_stopping=True,        # activa early stopping
    #max_epochs=tcn_params_60_50["max_epochs"],                           # podés subirlo un poco
    #patience=tcn_params_60_50["patience"],                             # paciencia del early stopping
    verbose=False,                           # logs de entrenamiento

    # Si tus ventanas están aplanadas (2D), pasá su forma original:
    # enforce_3d_shape=(window_size, n_features)
    enforce_3d_shape=(90, len(features_to_60)+len(features_base)), # ejemplo si window_size=90

    # Hooks genéricos:
    evaluate_fn=evaluate_model,              # misma función de métricas
    print_metrics_fn=print_metrics           # tu función para mostrar métricas
)
# Tiempo:  291s

Entrenando modelo TCN_60_subsampleado_50% (resample=50%)...
Métricas de TCN_60_subsampleado_50%:

	 RMSE:	 0.004650
	  MAE:	 0.002713
	   R2:	 0.082093
	SMAPE:	 116.437187
	DirAcc:	 0.693018


#### 4.2.3. Con ventanas completas

In [ ]:
tcn_params_60_100["use_weight_norm"] = False

In [90]:
tcn_60_100 = run_tcn_experiment(
    metrics_flag=False,
    model_key="TCN_60_100%",
    X_train_scaled=X_train_30_scaled, y_train=y_train_30,
    X_valid_scaled=X_valid_30_scaled, y_valid=y_valid_30,
    resample_rate=1.0,
    tcn_params=tcn_params_30_100,
    tcn_metrics_df=tcn_metrics,
    use_internal_early_stopping=True,
    max_epochs=tcn_params_30_100["max_epochs"],
    patience=tcn_params_30_100["patience"],
    verbose=tcn_params_30_100["verbose"],
    enforce_3d_shape=(90, len(features_to_30)+len(features_base)),
    evaluate_fn=evaluate_model,
    print_metrics_fn=print_metrics
)

Entrenando modelo TCN_60_100% (resample=100%)...
Métricas de TCN_60_100%:

	 RMSE:	 0.017482
	  MAE:	 0.012610
	   R2:	 -13.175317
	SMAPE:	 161.303594
	DirAcc:	 0.526475


In [91]:
tcn_metrics

,RMSE,MAE,R2,SMAPE,DirAcc
TCN_30_subsampleado_30%,0.003655,0.002355,-0.306041,130.023726,0.611708
TCN_30_subsampleado_50%,0.003750,0.002498,-0.343409,131.578467,0.601164
TCN_30_100%,0.012346,0.009199,-13.233888,158.551167,0.581471
TCN_60_subsampleado_30%,0.005542,0.003168,-0.445459,126.158065,0.643705
TCN_60_subsampleado_50%,0.004650,0.002713,0.082093,116.437187,0.693018
TCN_60_100%,0.017482,0.012610,-13.175317,161.303594,0.526475
TCN_90_subsampleado_30%,0.005707,0.003180,0.063686,111.166983,0.725662
TCN_90_subsampleado_50%,0.005412,0.002896,0.183344,104.811059,0.751480
TCN_90_100%,0.009756,0.007167,-1.746164,143.959714,0.588905


### 4.3. Entrenamiento 90min

#### 4.3.1. Con 30% de dataset

In [92]:
tcn_90_30 = run_tcn_experiment(
    metrics_flag=metrics,
    model_key="TCN_90_subsampleado_30%",
    X_train_scaled=X_train_90_scaled, y_train=y_train_90,
    X_valid_scaled=X_valid_90_scaled, y_valid=y_valid_90,
    resample_rate=0.3,                       # 30% del dataset
    tcn_params=tcn_params_90_30,           # hiperparámetros base
    tcn_metrics_df=tcn_metrics,              # DataFrame donde guardar métricas
    use_internal_early_stopping=True,        # activa early stopping
    #max_epochs=tcn_params_90_30["max_epochs"],                           # podés subirlo un poco
    #patience=tcn_params_90_30["patience"],                             # paciencia del early stopping
    verbose=False,                           # logs de entrenamiento

    # Si tus ventanas están aplanadas (2D), pasá su forma original:
    # enforce_3d_shape=(window_size, n_features)
    enforce_3d_shape=(90, len(features_to_90)+len(features_base)), # ejemplo si window_size=90

    # Hooks genéricos:
    evaluate_fn=evaluate_model,              # misma función de métricas
    print_metrics_fn=print_metrics           # tu función para mostrar métricas
)
#Time: >3mins

Entrenando modelo TCN_90_subsampleado_30% (resample=30%)...
Métricas de TCN_90_subsampleado_30%:

	 RMSE:	 0.006020
	  MAE:	 0.003253
	   R2:	 0.038149
	SMAPE:	 112.926745
	DirAcc:	 0.711307


#### 4.3.2. Con 50% de dataset

In [93]:
tcn_90_50 = run_tcn_experiment(
    metrics_flag=metrics,
    model_key="TCN_90_subsampleado_50%",
    X_train_scaled=X_train_90_scaled, y_train=y_train_90,
    X_valid_scaled=X_valid_90_scaled, y_valid=y_valid_90,
    resample_rate=0.5,                       # 30% del dataset
    tcn_params=tcn_params_90_50,           # hiperparámetros base
    tcn_metrics_df=tcn_metrics,              # DataFrame donde guardar métricas
    use_internal_early_stopping=True,        # activa early stopping
    #max_epochs=tcn_params_90_50["max_epochs"],                           # podés subirlo un poco
    #patience=tcn_params_90_50["patience"],                             # paciencia del early stopping
    verbose=False,                           # logs de entrenamiento

    # Si tus ventanas están aplanadas (2D), pasá su forma original:
    # enforce_3d_shape=(window_size, n_features)
    enforce_3d_shape=(90, len(features_to_90)+len(features_base)), # ejemplo si window_size=90

    # Hooks genéricos:
    evaluate_fn=evaluate_model,              # misma función de métricas
    print_metrics_fn=print_metrics           # tu función para mostrar métricas
)
# Tiempo:  > 4min

Entrenando modelo TCN_90_subsampleado_50% (resample=50%)...
Métricas de TCN_90_subsampleado_50%:

	 RMSE:	 0.005326
	  MAE:	 0.003150
	   R2:	 0.124247
	SMAPE:	 108.833383
	DirAcc:	 0.737815


#### 4.3.3. Con ventanas completas

In [94]:
tcn_90_100 = run_tcn_experiment(
    metrics_flag=metrics,
    model_key="TCN_90_100%",
    X_train_scaled=X_train_90_scaled, y_train=y_train_90,
    X_valid_scaled=X_valid_90_scaled, y_valid=y_valid_90,
    resample_rate=1.0,                       # 30% del dataset
    tcn_params=tcn_params_90_100,           # hiperparámetros base
    tcn_metrics_df=tcn_metrics,              # DataFrame donde guardar métricas
    use_internal_early_stopping=True,        # activa early stopping
    #max_epochs=tcn_params_90_100["max_epochs"],                           # podés subirlo un poco
    #patience=tcn_params_90_100["patience"],                             # paciencia del early stopping
    verbose=False,                           # logs de entrenamiento

    # Si tus ventanas están aplanadas (2D), pasá su forma original:
    # enforce_3d_shape=(window_size, n_features)
    enforce_3d_shape=(90, len(features_to_90)+len(features_base)), # ejemplo si window_size=90

    # Hooks genéricos:
    evaluate_fn=evaluate_model,              # misma función de métricas
    print_metrics_fn=print_metrics           # tu función para mostrar métricas
)

#Time 342 segundos.

Entrenando modelo TCN_90_100% (resample=100%)...
Métricas de TCN_90_100%:

	 RMSE:	 0.017074
	  MAE:	 0.013316
	   R2:	 -7.412191
	SMAPE:	 157.176582
	DirAcc:	 0.591792


## 5. Recuperación de métricas

In [95]:
tcn_metrics

,RMSE,MAE,R2,SMAPE,DirAcc
TCN_30_subsampleado_30%,0.003655,0.002355,-0.306041,130.023726,0.611708
TCN_30_subsampleado_50%,0.003750,0.002498,-0.343409,131.578467,0.601164
TCN_30_100%,0.012346,0.009199,-13.233888,158.551167,0.581471
TCN_60_subsampleado_30%,0.005542,0.003168,-0.445459,126.158065,0.643705
TCN_60_subsampleado_50%,0.004650,0.002713,0.082093,116.437187,0.693018
TCN_60_100%,0.017482,0.012610,-13.175317,161.303594,0.526475
TCN_90_subsampleado_30%,0.006020,0.003253,0.038149,112.926745,0.711307
TCN_90_subsampleado_50%,0.005326,0.003150,0.124247,108.833383,0.737815
TCN_90_100%,0.017074,0.013316,-7.412191,157.176582,0.591792


In [95]:
save_metrics(tcn_metrics, "4_6_tcn_metrics_1")

Métricas guardadas en /content/drive/MyDrive/neural_profit/4_model_training/4_6_tcn_metrics_1.parquet


Este punto existe para garantizar reproducibilidad y continuidad del análisis sin reentrenar modelos cuando se pierden las métricas. Actúa como fallback: reconstruye la tabla de métricas de Random Forest a partir de valores ya validados y la persiste nuevamente, evitando el re-entrenamiento de los modelos.

Así, se mantiene la consistencia de resultados y la trazabilidad de comparaciones y conclusiones, incluso si el archivo original fue eliminado, corrompido o el entorno de ejecución cambió.

In [ ]:
def generate_metrics_mlp():
  '''
  Ejecutar está función solo en caso de perder las métricas
  El objetivo es no volver a correr los entrenamientos
  '''
  mlp_metrics = pd.DataFrame(columns=["RMSE", "MAE", "R2", "SMAPE", "DirAcc"])

  mlp_30_30 = {
      "RMSE":	 0.002537,
      "MAE":	 0.001655,
      "R2":	 0.333589,
      "SMAPE":	 120.453559,
      "DirAcc":	 0.671852
  }

  mlp_30_50 = {
      "RMSE": 0.002543,
      "MAE": 0.001607,
      "R2": 0.397152,
      "SMAPE": 119.067749,
      "DirAcc": 0.699514
  }

  mlp_60_30 = {
      "RMSE": 0.003266,
      "MAE": 0.001943,
      "R2": 0.505579,
      "SMAPE": 101.422228,
      "DirAcc": 0.767442
  }

  mlp_60_50 = {
    'RMSE': 0.003372370312304722,
    'MAE': 0.001954590335922684,
    'R2': 0.4532909519346531,
    'SMAPE': 101.99432110111918,
    'DirAcc': 0.7678872155126786
 }

  mlp_90_30 = {
      'RMSE': 0.0036072614892137035,
      'MAE': 0.0022671873954739,
      'R2': 0.6190330484920656,
      'SMAPE': 93.33966545801738,
      'DirAcc': 0.7947072975140337
  }

  mlp_90_50 = {
    'RMSE': 0.00425000104407735,
    'MAE': 0.0022730454470971714,
    'R2': 0.4603383303081291,
    'SMAPE': 93.93283824100307,
    'DirAcc': 0.8030601934273204
      }

  mlp_metrics.loc['MLP_30_subsampleado_30%'] = mlp_30_30
  mlp_metrics.loc['MLP_30_subsampleado_50%'] = mlp_30_50

  mlp_metrics.loc['MLP_60_subsampleado_30%'] = mlp_60_30
  mlp_metrics.loc['MLP_60_subsampleado_50%'] = mlp_60_50

  mlp_metrics.loc['MLP_90_subsampleado_30%'] = mlp_90_30
  mlp_metrics.loc['MLP_90_subsampleado_50%'] = mlp_90_50


  save_metrics(mlp_metrics, "4_5_mlp_metrics")
  return mlp_metrics

In [ ]:
#Ejecutar esta función solo en caso de perder las métricas de entrenamiento
#mlp_metrics = generate_metrics_mlp()

Métricas guardadas en /content/drive/MyDrive/neural_profit/4_model_training/4_3_lgbm_metrics.parquet


In [ ]:
tcn_metrics

,RMSE,MAE,R2,SMAPE,DirAcc
MLP_30_subsampleado_30%,0.002537,0.001655,0.333589,120.453559,0.671852
MLP_30_subsampleado_50%,0.002543,0.001607,0.397152,119.067749,0.699514
MLP_60_subsampleado_30%,0.003266,0.001943,0.505579,101.422228,0.767442
MLP_60_subsampleado_50%,0.003372,0.001955,0.453291,101.994321,0.767887
MLP_90_subsampleado_30%,0.003607,0.002267,0.619033,93.339665,0.794707
